# Les décorateurs

In [15]:
import functools

def sandwich(content):
    @functools.wraps(content)
    def wrapper(*args, **kargs):
        print("/TTTTTTTT\\")
        price = content(*args, **kargs)
        print("\\________/")

        return price + 1.5
        
    return wrapper

In [11]:
def demo_parisien():
    """
    Une doc
    """
    print("  Jambon ")
    print("  Beurre ")

sandwich(demo_parisien)


<function __main__.sandwich.<locals>.wrapper(*args, **kargs)>

In [12]:
demo_parisien.__doc__

'\nUne doc\n'

In [16]:
@sandwich
def parisien():
    """
    Une doc
    """
    print("  Jambon ")
    print("  Beurre ")
    return 5

@sandwich
def lyonnais():
    print("  Rosette ")
    print("  Beurre ")
    return 6

@sandwich
def kebab(salade_tomates_oingons:bool=True):
    if salade_tomates_oingons:
        print(" 🍅 ")
    print("  viande ")

    return 7
    

In [17]:
parisien.__name__

'parisien'

In [18]:
parisien.__doc__

'\nUne doc\n'

In [ ]:
kebab(salade_tomates_oingons=False)

In [ ]:
print(parisien())

In [ ]:
lyonnais()

## Les décorateurs paramétrés

In [36]:

def sandwich(_func=None, *, hot=False):
    def inner_deco(content):
        def wrapper(*args, **kargs):
            print("/TTTTTTTT\\")
            price = content(*args, **kargs)
            print("\\________/")

            if hot:
                print("ça cuit")
            return price + 1.5
            
        return wrapper

    if _func is None:
        return inner_deco
    else:
        return inner_deco(_func)

In [39]:
@sandwich
def panini():
    print(" MOzarella ")
    return 6

In [40]:
panini()

/TTTTTTTT\
 MOzarella 
\________/


7.5

## Exercices

### Premier exercice

In [6]:
import time

def time_it(func):
    def wrapper(*args, **kargs):
        start = time.time()
        func(*args, **kargs)
        end = time.time()

        print(f'{end - start}, secondes se sont écoulées')

    return wrapper

@time_it
def pow_2():
    for x in range(1_000_000):
        y = x ** 2

def pow_2_raw():
    for x in range(1_000_000):
        y = x ** 2



In [ ]:
pow_2()

In [ ]:
%timeit pow_2()

In [ ]:
%timeit pow_2_raw()

In [ ]:
import time

start = time.time()

for x in range(1_000_000):
    y = x ** 2

end = time.time()

print(end - start, "secondes se sont écoulées")

### Deuxième exercice

In [ ]:
def count_calls(func):
    count = 0

    def wrapper():
        nonlocal count
        func()
        count += 1

    def get_counts():
        return count

    wrapper.nbcalls = get_counts

    return wrapper

@count_calls
@time_it
def some_func():
    print("func called")

print(some_func.nbcalls())
some_func()
some_func()
print(some_func.nbcalls())

### Troisième exercice

In [7]:
import time

def cached_long_call(func):

    cached_call:tuple = (None, None)
    
    def inner_long_call(value:int):
        nonlocal cached_call
        if value == cached_call[0]:
            result = cached_call[1]
        else:
            result = func(value)
            cached_call = (value, result)

        return result

    return inner_long_call


@time_it
@cached_long_call
def long_call(value:int):
    time.sleep(2)
    return value**2

In [10]:
long_call(43)

2.0050368309020996, secondes se sont écoulées


In [ ]:
from collections import OrderedDict

def cached_long_call_dict(func):

    cached_calls = OrderedDict()
    MAX_CACHE = 3
    
    def inner_long_call(value:int):
        try:
            result = cached_calls[value]
            cached_calls.move_to_end(value)
        except KeyError:
            result = func(value)
            cached_calls[value] = result
            if len(cached_calls) > MAX_CACHE:
                cached_calls.popitem(last=False)

        return result

    return inner_long_call

### Quatrième exercice

In [16]:
notifications = []

def register(func):
    notifications.append(func)
    return func

@register
def notify_mail():
    print("notification on mail")

def notify_sms():
    print("notification on message")

@register
def notify_push():
    print("notification on push service")

def send_notifications():
    for notification in notifications:
        notification()

send_notifications()

notification on mail
notification on push service
